### nxsut generator — v3.0

MARIO-native pipeline: parses EXIOBASE Hybrid v3.3.18, updates the electricity supply mix from EMBER (via the nxbase query API), pools electricity trade behind a supply/need pass-through layer, and updates the trade mix from the **open ENTSO-E** scheduled-exchange set (via the nxbase query API). Fully open input chain — the first publishable nxsut version. Supersedes the retired `v2.1` (which used the proprietary Electricity Maps mix on the same MARIO-native pipeline).

ENTSO-E covers the European countries; every other EXIOBASE region (US, CN, JP, … and the RoW aggregates) is filled domestic-only — as are the origin-only regions ENTSO-E returns as an all-zero destination column (e.g. Luxembourg, Malta: their real import dependence disappears in this version, a known residual pending an ENTSO-E control-area fetch).

Set `user` and `year` in the first cell, then run top to bottom.

In [8]:
import mario
import yaml
import os

with open('paths.yml', 'r') as file:  # open the yml file
    paths = yaml.safe_load(file)

user = 'LR'   # change this to your username
year = 2023   # change this to the year you want to build

paths = paths[user]
import warnings
warnings.filterwarnings("ignore")

from support import nxbase_client as nxc
nxbase_api = paths.get('nxbase_api', nxc.DEFAULT_API)


Parse the raw EXIOBASE database, aggregate electricity to EMBER resolution, then add the explicit steel & H2 production routes (Ghezzi et al. 2026 recipe, fetched from the nxbase query API). `add_sectors` runs **after** `aggregate_ee`: the routes attach to the single aggregated grid `Electricity`, and the pre-existing EMBER electricity activities keep their supply coefficients. `meta.source` enables MARIO's EXIOBASE Rest-of-World member-country expansion when using EMBER.

In [9]:
db = mario.parse_from_txt(paths['raw'], table='SUT', mode='flows')
db.meta.source = 'EXIOBASE Hybrid 3.3.18'
db.aggregate('support/aggregate_ee.xlsx', ignore_nan=True)

# steel/H2 sectors from the nxbase recipe -- add AFTER aggregate_ee so the new
# routes attach to the aggregated grid 'Electricity', and the pre-existing EMBER
# electricity activities keep their supply coefficients (running add_sectors
# before aggregate would drop them from the 's' block -> update_supply_mix fails).
nxc.build_add_sectors_master('support/add_sectors/Master_steel_h2.xlsx', '_steel_master.xlsx', api_url=nxbase_api)
db.read_add_sectors_excel('_steel_master.xlsx', read_inventories=True)
db.add_sectors()

INFO Parser: txt reading SUT flows from /Users/lorenzorinaldi/Library/CloudStorage/OneDrive-SharedLibraries-eNextGen/eNextAll - Documents/Databases/Exiobase Hybrid 3.3.18 with VA/flows in matrix mode (txt/csv).
INFO Parser: Reading flows from txt files.
INFO Parser: Reading files finished.
INFO Parser: Investigating possible identifiable errors.
INFO Parser: parsing database finished.
INFO Parser: state payload ready with 9 canonical blocks.
INFO Parser: txt state ready for SUT.
INFO Metadata: initialized.
WARNING nan values for the aggregation of Activity for following items ignored
['Cultivation of paddy rice', 'Cultivation of wheat', 'Cultivation of cereal grains nec', 'Cultivation of vegetables, fruit, nuts', 'Cultivation of oil seeds', 'Cultivation of sugar cane, sugar beet', 'Cultivation of plant-based fibers', 'Cultivation of crops nec', 'Cattle farming', 'Pigs farming', 'Poultry farming', 'Meat animals nec', 'Animal products nec', 'Raw milk', 'Wool, silk-worm cocoons', 'Manure 

### Furnace-gas emission reallocation (ExioSteel method)

Two fictitious activities take the blast/oxygen furnace gas by-products (the steel sector's supply of them is zeroed), and the steel sector's coefficients are recomputed on its steel supply alone (`U/S_main`, not `U/X`) → footprint per tonne of *actual steel*, not diluted by the co-product gases. Runs **after** the Ghezzi `add_sectors`, **before** the supply mix.

In [ ]:
# Furnace-gas emission reallocation (ExioSteel method): two fictitious gas-
# production activities take the blast/oxygen furnace gas by-products (steel
# supply of them zeroed), and the steel sector's U/V/E are recomputed on its
# steel supply alone (/ S_main, not / X) -> footprint per tonne of actual steel.
db.read_add_sectors_excel('support/add_sectors/blastfurnacegas.xlsx', read_inventories=True)
db.add_sectors()

STEEL_ACT = 'Manufacture of basic iron and steel and of ferro-alloys and first products thereof'
STEEL_COM = 'Basic iron and steel and of ferro-alloys and first products thereof'
s, u, v, e = db.s, db.u, db.v, db.e
by_product = list(db.add_sectors_master['Commodity'].unique())
gas_of_act = {a: db.add_sectors_master.loc[db.add_sectors_master['Activity'] == a, 'Commodity'].values[0]
              for a in db.new_activities}
for region in db.get_index('Region'):
    s_byprod = (s.loc[(region, 'Activity', STEEL_ACT), (region, 'Commodity', by_product)] * 0).to_frame().T
    s.update(s_byprod)
    for new_act, commodity in gas_of_act.items():
        s.loc[(region, 'Activity', new_act), (region, 'Commodity', commodity)] = 1
    S_main = db.S.loc[(region, 'Activity', STEEL_ACT), (region, 'Commodity', STEEL_COM)]
    u.update(db.U.loc[:, (region, 'Activity', STEEL_ACT)] / S_main.sum())
    v.update(db.V.loc[:, (region, 'Activity', STEEL_ACT)] / S_main.sum())
    e.update(db.E.loc[:, (region, 'Activity', STEEL_ACT)] / S_main.sum())

z = db.z
z.update(s)
z.update(u)
db.update_scenarios('baseline', z=z, v=v, e=e)
db.reset_to_coefficients('baseline')
print('BFG/OFG reallocation applied; new activities:', list(db.new_activities))

Supply mix from nxbase (query API) — see `support/nxbase_client.py`. MARIO reads the reduced EMBER snapshot from a transient file, regenerated every run.

In [10]:
from support import nxbase_client as nxc

nxbase_api = paths.get('nxbase_api', nxc.DEFAULT_API)
print(nxc.get_provenance(nxbase_api, ['EMBER Yearly Electricity Data 2025']))

ember_snapshot_path = 'support/_nxbase_ember_snapshot.csv'
nxc.get_ember_snapshot(nxbase_api).to_csv(ember_snapshot_path, index=False)

db.update_supply_mix(
    "electricity",
    scenario = 'baseline',
    year = year,
    ember_path = ember_snapshot_path,
)

nxbase version: 0.1.0 (api: http://127.0.0.1:8000)
INFO Resolver: resolving ea for baseline.
INFO Resolver: trying ea via extract.
INFO Resolver: resolved ea via extract.
INFO Resolver: resolving ec for baseline.
INFO Resolver: trying ec via extract.
INFO Resolver: resolved ec via extract.
INFO Resolver: resolving va for baseline.
INFO Resolver: trying va via extract.
INFO Resolver: resolved va via extract.
INFO Resolver: resolving vc for baseline.
INFO Resolver: trying vc via extract.
INFO Resolver: resolved vc via extract.
INFO Resolver: resolving Ya for baseline.
INFO Resolver: trying Ya via extract.
INFO Resolver: resolved Ya via extract.
INFO Resolver: resolving Yc for baseline.
INFO Resolver: trying Yc via extract.
INFO Resolver: resolved Yc via extract.
INFO Databases: reset to coefficients.
INFO Electricity mix: aggregating database sectors to EMBER groups and redistributing with the current internal composition.
INFO Electricity mix: no EMBER data for 2023 in some covered regi

Pool the trade of the selected commodities. MARIO adds the `" supply"` / `" need"` pass-through layer and stores the observed trade shares in the supply block market shares. The suffixes match the `NXS2` namespace rows in nxbase.

In [11]:
traded_commodities = ['Electricity']
db.pool_trade(traded_commodities, supply_suffix=" supply", need_suffix=" need")

INFO Resolver: resolving Z for baseline.
INFO Resolver: trying Z via concat.
INFO Resolver: resolved Xc via formula build_sut_Xc_from_u_s_Yc (compute_method=inverse, runtime=solve).
INFO Resolver: resolved Z via concat.
INFO Resolver: resolving Y for baseline.
INFO Resolver: trying Y via concat.
INFO Resolver: resolved Y via concat.
INFO Resolver: resolving V for baseline.
INFO Resolver: trying V via concat.
INFO Resolver: resolved Xc via formula build_sut_Xc_from_u_s_Yc (compute_method=inverse, runtime=solve).
INFO Resolver: resolved V via concat.
INFO Resolver: resolving E for baseline.
INFO Resolver: trying E via concat.
INFO Resolver: resolved Xc via formula build_sut_Xc_from_u_s_Yc (compute_method=inverse, runtime=solve).
INFO Resolver: resolved E via concat.
INFO pool_trade: pooled trade layer added for ['Electricity'].


**Update trade mixes** from the open ENTSO-E scheduled-exchange set (nxbase query API). One origins-by-destinations matrix per commodity; a positive column sum — not mere presence — decides "covered" (ENTSO-E returns some destinations as an all-zero column, which `update_trade_mix` would otherwise reject). `rescale=True` normalizes each destination mix while preserving destination column totals.

In [12]:
scenario = 'entsoe_trades'
if scenario not in db.scenarios:
    db.clone_scenario('baseline', scenario)

regions = list(db.get_index('Region'))
for commodity in traded_commodities:
    pooled = db.meta.pooled_trade_map[commodity]
    trades = nxc.get_trade_matrix(
        nxbase_api, year=year, commodity=commodity,
        source=f"ENTSO-E electricity import mix {year}",
    )
    trade_dict = {}
    for dest in regions:
        col = trades[dest] if dest in trades.columns else None
        trade_dict[dest] = (
            col.dropna().to_dict() if (col is not None and col.sum() > 0) else {dest: 1.0}
        )
    db.update_trade_mix(
        trade_dict,
        items = pooled['supply'],
        commodities = pooled['need'],
        scenario = scenario,
        rescale = True,
    )

INFO Resolver: resolving u for entsoe_trades.
INFO Resolver: trying u via extract.
INFO Resolver: trying u via formula build_sut_u_from_U_Xa.
INFO Resolver: resolved u via formula build_sut_u_from_U_Xa.
INFO Resolver: resolving s for entsoe_trades.
INFO Resolver: trying s via extract.
INFO Resolver: trying s via formula build_sut_s_from_S_Xc.
INFO Resolver: resolved s via formula build_sut_s_from_S_Xc.
INFO Resolver: resolving ea for entsoe_trades.
INFO Resolver: trying ea via extract.
INFO Resolver: trying ea via formula build_sut_ea_from_Ea_Xa.
INFO Resolver: resolved ea via formula build_sut_ea_from_Ea_Xa.
INFO Resolver: resolving ec for entsoe_trades.
INFO Resolver: trying ec via extract.
INFO Resolver: trying ec via formula build_sut_ec_from_Ec_Xc.
INFO Resolver: resolved ec via formula build_sut_ec_from_Ec_Xc.
INFO Resolver: resolving va for entsoe_trades.
INFO Resolver: trying va via extract.
INFO Resolver: trying va via formula build_sut_va_from_Va_Xa.
INFO Resolver: resolved v

Export v3.0.

In [13]:
v30_path = os.path.join(paths['export'], "v3.0", str(year))
os.makedirs(v30_path, exist_ok=True)
db.to_txt(path = v30_path, scenario = scenario)

INFO Export: writing txt database for entsoe_trades in matrix mode.
INFO Resolver: resolving U for entsoe_trades.
INFO Resolver: trying U via extract.
INFO Resolver: trying U via formula build_sut_U_from_u_Xa.
INFO Resolver: resolved Xc via formula build_sut_Xc_from_u_s_Yc (compute_method=inverse, runtime=solve).
INFO Resolver: resolved U via formula build_sut_U_from_u_Xa.
INFO Resolver: resolving S for entsoe_trades.
INFO Resolver: trying S via extract.
INFO Resolver: trying S via formula build_sut_S_from_s_Xc.
INFO Resolver: resolved S via formula build_sut_S_from_s_Xc.
INFO Resolver: resolving Va for entsoe_trades.
INFO Resolver: trying Va via extract.
INFO Resolver: trying Va via formula build_sut_Va_from_va_Xa.
INFO Resolver: resolved Va via formula build_sut_Va_from_va_Xa.
INFO Resolver: resolving Vc for entsoe_trades.
INFO Resolver: trying Vc via extract.
INFO Resolver: trying Vc via formula build_sut_Vc_from_vc_Xc.
INFO Resolver: resolved Vc via formula build_sut_Vc_from_vc_Xc.

---
## Footprint comparison: v2.0 (legacy Electricity Maps) vs v3.0 (open ENTSO-E)

Non-EU / RoW regions are domestic-only in both pipelines, so their electricity footprints should be near-identical; any real change concentrates in the ENTSO-E-covered European countries whose import mix actually differs between the two datasets. Requires a `v2.0/<year>` export already on disk (`gen_v2.ipynb`).

In [14]:
import pandas as pd

db_old = mario.parse_from_txt(
    path = os.path.join(paths['export'], "v2.0", str(year), "flows"),
    mode = "flows",
    table = 'SUT',
)

gwp = {
    "Carbon dioxide, fossil (air - Emiss)": 1.0,
    "CH4 (air - Emiss)": 25.0,
    "N2O (air - Emiss)": 298.0,
}

def ghg_footprint(f):
    f = f.loc[list(gwp), :].T
    return sum(f[substance] * factor for substance, factor in gwp.items())

f_v30 = ghg_footprint(db.query('f', scenarios=scenario))
f_old = ghg_footprint(db_old.f)

# Align the legacy pooled labels to the new naming convention.
item_renames = {}
for commodity in traded_commodities:
    pooled = db.meta.pooled_trade_map[commodity]
    item_renames[f"{commodity} supply"] = pooled['supply']
    item_renames[f"{commodity} need"] = pooled['need']
f_old = f_old.rename(index=item_renames, level='Item')

comp = pd.concat([f_old.rename('v2.0'), f_v30.rename('v3.0')], axis=1)
comp['Delta%'] = 100 * (comp['v3.0'] / comp['v2.0'] - 1)

INFO Parser: txt reading SUT flows from /Users/lorenzorinaldi/Library/CloudStorage/OneDrive-SharedLibraries-eNextGen/eNextAll - Documents/Databases/nxsut/v2.0/2023/flows in matrix mode (txt/csv).
INFO Parser: Reading flows from txt files.
INFO Parser: Reading files finished.
INFO Parser: Investigating possible identifiable errors.
INFO Parser: parsing database finished.
INFO Parser: state payload ready with 9 canonical blocks.
INFO Parser: txt state ready for SUT.
INFO Metadata: initialized.
INFO Resolver: resolving f for entsoe_trades.
INFO Resolver: trying f via concat.
INFO Resolver: resolved fa via formula build_sut_fa_from_ea_s_u (compute_method=inverse, runtime=solve).
INFO Resolver: resolved fc via formula build_sut_fc_from_ea_s_u (compute_method=inverse, runtime=solve).
INFO Resolver: resolved f via concat.
INFO Resolver: resolving f for baseline.
INFO Resolver: trying f via concat.
INFO Resolver: resolved waa via formula build_sut_waa_from_s_u (compute_method=inverse, runtime=

In [15]:
# Electricity-need GHG intensity per region, split EU (ENTSO-E-covered) vs non-EU.
NON_EU = {'US', 'CN', 'JP', 'KR', 'BR', 'IN', 'MX', 'RU', 'AU', 'ID', 'ZA',
          'CA', 'TW', 'WA', 'WL', 'WE', 'WF', 'WM'}
need = db.meta.pooled_trade_map['Electricity']['need']
ele = comp.loc[(slice(None), 'Commodity', need), :].copy()
ele.index = ele.index.get_level_values('Region')
ele['group'] = ['non-EU/RoW' if r in NON_EU else 'EU/ENTSO-E' for r in ele.index]

summary = ele.groupby('group')['Delta%'].agg(
    n='count',
    mean_abs=lambda s: s.abs().mean(),
    max_abs=lambda s: s.abs().max(),
).round(3)
print("Electricity-need footprint, v3.0 vs v2.0 — |Delta%| by group:")
print(summary)
print("\nLargest |Delta%| overall (electricity need):")
display(ele.reindex(ele['Delta%'].abs().sort_values(ascending=False).index).round(3).head(20))

Electricity-need footprint, v3.0 vs v2.0 — |Delta%| by group:
             n  mean_abs  max_abs
group                            
EU/ENTSO-E  31     7.909   72.011
non-EU/RoW  17     2.399   21.772

Largest |Delta%| overall (electricity need):


,v2.0,v3.0,Delta%,group
Region,,,,
MT,25.338,7.092,-72.011,EU/ENTSO-E
LU,83.940,33.096,-60.571,EU/ENTSO-E
FI,32.401,41.135,26.954,EU/ENTSO-E
WE,156.759,122.629,-21.772,non-EU/RoW
CH,19.108,16.023,-16.147,EU/ENTSO-E
SK,85.968,72.789,-15.330,EU/ENTSO-E
HR,73.836,64.120,-13.160,EU/ENTSO-E
LV,79.126,88.637,12.020,EU/ENTSO-E
WF,91.351,99.038,8.415,non-EU/RoW
